# LLMOps Assignment 2 — Optimized implementation

**Model:** `llama3.1:8b` via local Ollama  
**Efficiency:** sequential requests, temperature 0, 256-token output cap, no retries, no plots, compact CSV outputs.

Before running, start Ollama and download the model:
```bash
ollama serve
ollama pull llama3.1:8b
```

Install dependencies:
```bash
pip install ollama mlflow pandas
```


In [ ]:
# LLMOps Assignment 2 — Optimized local run with Ollama llama3.1:8b
# Run Ollama first: ollama serve  (if not already running)
# Pull model once: ollama pull llama3.1:8b

import json, time, math
from datetime import datetime, timezone
from pathlib import Path

import ollama
import pandas as pd
import mlflow
from mlflow import MlflowClient

MODEL = "llama3.1:8b"
MONTHLY_REQUESTS = 10_000
PRICE_PER_1K_INPUT = 0.0   # Local Ollama: no per-token API charge
PRICE_PER_1K_OUTPUT = 0.0
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

QUESTIONS = [
    "Explain what machine learning is in two sentences.",
    "List three practical uses of artificial intelligence.",
    "What is data preprocessing and why is it important?",
    "Explain precision and recall with a simple example.",
    "Give three ways to reduce overfitting in a machine learning model."
]

PROMPTS = {
    "V1": "Answer this question clearly and briefly:\n{question}",
    "V2": ("You are a helpful academic assistant. Answer the question accurately. "
           "Use a short heading and 2-4 concise bullet points. "
           "If uncertain, say so.\nQuestion: {question}"),
    "V3": ("Return ONLY valid JSON with keys answer (string) and key_points (array of strings). "
           "Do not include Markdown fences or text outside JSON.\nQuestion: {question}")
}

client = ollama.Client(host="http://localhost:11434")
# Fail early instead of wasting time in the evaluation loop.
if MODEL not in [m.model.split(":")[0] + (":" + m.model.split(":")[1] if ":" in m.model else "")
                 for m in client.list().models]:
    print(f"Model {MODEL} may not be downloaded. Run: ollama pull {MODEL}")

def invoke(prompt, json_mode=False):
    start = time.perf_counter()
    kwargs = dict(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 256}
    )
    if json_mode:
        kwargs["format"] = "json"
    result = client.chat(**kwargs)
    latency = time.perf_counter() - start
    return (result.message.content or "").strip(), latency, \
           int(result.prompt_eval_count or 0), int(result.eval_count or 0)

rows = []
mlflow.set_experiment("eduassist_llmops_assignment")
client_mlflow = MlflowClient()

for version, template in PROMPTS.items():
    with mlflow.start_run(run_name=f"{version}_{MODEL.replace(':','_')}"):
        for i, question in enumerate(QUESTIONS, 1):
            prompt = template.format(question=question)
            answer, latency, in_tok, out_tok = invoke(prompt, json_mode=(version == "V3"))
            valid_json = False
            if version == "V3":
                try:
                    obj = json.loads(answer)
                    valid_json = isinstance(obj, dict) and "answer" in obj and "key_points" in obj
                except (json.JSONDecodeError, TypeError):
                    pass

            cost = (in_tok / 1000 * PRICE_PER_1K_INPUT) + \
                   (out_tok / 1000 * PRICE_PER_1K_OUTPUT)
            rows.append({
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "question_id": i,
                "model": MODEL,
                "prompt_version": version,
                "status": "success" if answer else "empty_response",
                "latency_seconds": round(latency, 4),
                "input_tokens": in_tok,
                "output_tokens": out_tok,
                "total_tokens": in_tok + out_tok,
                "cost_per_request": cost,
                "monthly_cost_10000": cost * MONTHLY_REQUESTS,
                "valid_json": valid_json if version == "V3" else None
            })

        df_run = pd.DataFrame([r for r in rows if r["prompt_version"] == version])
        mlflow.log_param("model", MODEL)
        mlflow.log_param("prompt_version", version)
        mlflow.log_param("requests", len(df_run))
        mlflow.log_metric("avg_latency_seconds", float(df_run.latency_seconds.mean()))
        mlflow.log_metric("avg_input_tokens", float(df_run.input_tokens.mean()))
        mlflow.log_metric("avg_output_tokens", float(df_run.output_tokens.mean()))
        mlflow.log_metric("avg_cost_per_request", float(df_run.cost_per_request.mean()))
        if version == "V3":
            mlflow.log_metric("valid_json", float(df_run.valid_json.mean()))
        mlflow.log_table(df_run, f"{version}_results.json")

results = pd.DataFrame(rows)
results.to_csv(OUT / "llmops_results.csv", index=False)

summary = (results.groupby("prompt_version")
           .agg(requests=("question_id", "count"),
                avg_latency_seconds=("latency_seconds", "mean"),
                avg_input_tokens=("input_tokens", "mean"),
                avg_output_tokens=("output_tokens", "mean"),
                avg_cost_per_request=("cost_per_request", "mean"),
                monthly_cost_10000=("monthly_cost_10000", "mean"))
           .reset_index())
summary.to_csv(OUT / "llmops_summary.csv", index=False)

print(summary.round(4).to_string(index=False))
print(f"\nSaved: {OUT/'llmops_results.csv'} and {OUT/'llmops_summary.csv'}")
print("Local Ollama inference has zero API-token charges; compute/electricity costs are not included.")


## Notes for the submission

- The notebook evaluates 5 questions against all 3 prompt versions (15 calls total).
- It logs latency, input/output/total tokens, status, cost estimate, and JSON validity for V3.
- Since Ollama runs locally, API-token cost is set to zero. This does **not** mean compute or electricity is free.
- MLflow experiment: `eduassist_llmops_assignment`. Prompt version is recorded as a run parameter for each version.
- Outputs are saved under `outputs/`.
